# Talmi EEG Data Preparation


Prepare the Talmi EEG study events for RecallDataset ingestion while tracking data quality decisions.


## Preprocessing Decisions
- Use the behavioral CSV to supply the full 20 study events for every subject/list before merging EEG features.
- Derive presentation order as TrialNumber - 2 and drop EEG rows with PresentationOrder == 99 whenever the trial already has 20 observations.
- Map rater codes so (1,1)→1.0, (2,2)→0.0, and (3,2)→0.5; store a lenient recalls array (0.5→recalled) and a strict ambiguous_recalls array (0.5→not recalled), both sorted by study position with trailing zeros.
- Preserve recorded item identities; otherwise generate deterministic placeholders and flag them in imputed_mask.
- Factorize study-level columns with fixed codes (ImgName via pd.factorize; Condition Negative→1, Neutral→2; ThreeConds Negative→1, Neutral→2, Category→3).
- Impute missing Early/Late LPP values using within-list, within-condition means, falling back to list means and then global condition means; mark those cells in lpp_imputed.
- Omit TrialNumber in the final dataset but retain subject, list (as column vectors), and listLength to satisfy the RecallDataset schema.
- Export per-position arrays (EarlyLPP, LateLPP, condition, cond3, imputed_mask, lpp_imputed, ambiguous_recalls) alongside the standard RecallDataset fields, accepting float types where needed.
- Confirm every participant contributes the modal nine lists so downstream code can rely on consistent trial counts.

## Setup


In [1]:
#| code-summary: Import modules for Talmi EEG profiling
import numpy as np
import pandas as pd
from pathlib import Path

from jaxcmr.helpers import save_dict_to_hdf5, find_project_root, generate_trial_mask


In [2]:
#| code-summary: Load behavioral and EEG CSV resources
DATA_DIR = Path('data/raw/talmieeg')
EEG_PATH = DATA_DIR / 'Single_Trial_Behavioural_and_EEG_Data_Z.csv'
BEHAVIOR_PATH = DATA_DIR / 'All_Included_Subjects.csv'

eeg_df = pd.read_csv(EEG_PATH)
behavior_df_raw = pd.read_csv(
    BEHAVIOR_PATH,
    header=None,
    names=[
        'Participant',
        'List',
        'TrialNumber',
        'ThreeCondsCode',
        'ConditionCode',
        'ThreeConds',
        'Recalled',
    ],
)


In [3]:
eeg_df.head()

,Participant,List,3Conds,Condition,TrialNumber,PresentationOrder,ImgName,Recalled,EarlyLPP,LateLPP
0,202,1,Negative,Negative,3,1,neg139,1,-3.95610,-1.84030
1,202,1,Neutral,Neutral,4,2,neut2,1,0.66242,-0.23096
2,202,1,Negative,Negative,5,3,neg138,1,-0.56351,-0.30037
3,202,1,Negative,Negative,6,4,neg132,0,0.72071,0.91095
4,202,1,Category,Neutral,7,5,cat124,0,-0.64235,0.51120


In [4]:
behavior_df_raw.head()


,Participant,List,TrialNumber,ThreeCondsCode,ConditionCode,ThreeConds,Recalled
0,202,1,3,1,1,Negative,1
1,202,1,4,2,2,Neutral,1
2,202,1,5,1,1,Negative,1
3,202,1,6,1,1,Negative,0
4,202,1,7,3,2,Category,0


## Helper Functions


In [5]:
#| code-summary: Define helpers for order alignment and imputation

TARGET_LIST_LENGTH = 20

def factorize_series(values: pd.Series, *, categories: list[str] | None = None) -> tuple[pd.Series, list[str]]:
    """Returns 1-based factor codes for provided series.

    Args:
      values: Column to convert into categorical codes.
      categories: Explicit category order assigning 1-based codes.

    Returns:
      Tuple of the 1-based codes aligned with the input index and the category labels.
    """
    if categories is None:
        codes, derived = pd.factorize(values, sort=True)
        return pd.Series(codes + 1, index=values.index), list(derived)
    mapping = {label: index + 1 for index, label in enumerate(categories)}
    mapped = values.map(mapping)
    if mapped.isna().any():
        unknown = values[mapped.isna()].unique().tolist()
        raise ValueError(f"Encountered unexpected categories: {unknown!r}")
    return mapped.astype(int), categories

def prepare_behavioral_dataframe(path: Path) -> pd.DataFrame:
    """Returns behavioral events with presentation order and recall flags.

    Args:
      path: Location of the behavioral CSV without headers.
    """
    df = pd.read_csv(
        path,
        header=None,
        names=[
            'Participant',
            'List',
            'TrialNumber',
            'ThreeCondsCode',
            'ConditionCode',
            'ThreeConds',
            'Recalled',
        ],
    )
    df['PresentationOrder'] = df['TrialNumber'] - 2
    df = df[df['PresentationOrder'].between(1, TARGET_LIST_LENGTH)].copy()
    df['Condition'] = df['ConditionCode'].map({1: 'Negative', 2: 'Neutral'})
    df['Recalled'] = df['Recalled'].astype(int)
    return df

def merge_behavior_and_eeg(behavior_df: pd.DataFrame, eeg_df: pd.DataFrame) -> pd.DataFrame:
    """Merge behavioral events with EEG features on presentation order.

    Args:
      behavior_df: Prepared behavioral rows with presentation order and recall flags.
      eeg_df: EEG rows including `ImgName`, `EarlyLPP`, and `LateLPP`.
    """
    usable_eeg = eeg_df[eeg_df['PresentationOrder'].between(1, TARGET_LIST_LENGTH)]
    return behavior_df.merge(
        usable_eeg[[
            'Participant',
            'List',
            'PresentationOrder',
            'ImgName',
            'EarlyLPP',
            'LateLPP',
        ]],
        on=['Participant', 'List', 'PresentationOrder'],
        how='left',
    )

def assign_img_placeholders(merged_df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing image names with deterministic placeholders.

    Args:
      merged_df: Rows after aligning behavior and EEG tables.
    """
    df = merged_df.copy()
    missing_mask = df['ImgName'].isna()
    df.loc[missing_mask, 'ImgName'] = df.loc[missing_mask].apply(
        lambda row: f"missing_{int(row.Participant)}_{int(row.List)}_{int(row.PresentationOrder)}",
        axis=1,
    )
    df['imputed_mask'] = missing_mask.astype(int)
    return df

def impute_lpp_by_condition(df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing LPP values using within-list condition averages.

    Args:
      df: Aligned study events containing LPP columns and condition labels.
    """
    result = df.copy()
    initial_missing = result[['EarlyLPP', 'LateLPP']].isna().any(axis=1)
    cond_means = result.groupby(['Participant', 'List', 'Condition'])[['EarlyLPP', 'LateLPP']].transform('mean')
    list_means = result.groupby(['Participant', 'List'])[['EarlyLPP', 'LateLPP']].transform('mean')
    global_means = result.groupby('Condition')[['EarlyLPP', 'LateLPP']].mean()
    for column in ['EarlyLPP', 'LateLPP']:
        result[column] = result[column].fillna(cond_means[column])
        result[column] = result[column].fillna(list_means[column])
        result[column] = result[column].fillna(result['Condition'].map(global_means[column]))
        result[column] = result[column].fillna(0.0)
    result['lpp_imputed'] = (
        initial_missing | result[['EarlyLPP', 'LateLPP']].isna().any(axis=1)
    ).astype(int)
    return result

def build_recall_dataset(completed_df: pd.DataFrame) -> dict[str, np.ndarray]:
    """Returns RecallDataset-aligned arrays including recall tracks.

    Args:
      completed_df: Study events with factorized identifiers and imputation flags.
    """
    ordered = completed_df.sort_values(['Participant', 'List', 'PresentationOrder'])
    trial_groups = list(ordered.groupby(['Participant', 'List'], sort=True))
    trial_count = len(trial_groups)
    subject = np.zeros((trial_count, 1), dtype=int)
    list_ids = np.zeros((trial_count, 1), dtype=int)
    list_length = np.full((trial_count, 1), TARGET_LIST_LENGTH, dtype=int)
    pres_itemnos = np.tile(np.arange(1, TARGET_LIST_LENGTH + 1, dtype=int), (trial_count, 1))
    pres_itemids = np.zeros((trial_count, TARGET_LIST_LENGTH), dtype=int)
    early_lpp = np.zeros((trial_count, TARGET_LIST_LENGTH), dtype=float)
    late_lpp = np.zeros((trial_count, TARGET_LIST_LENGTH), dtype=float)
    condition = np.zeros((trial_count, TARGET_LIST_LENGTH), dtype=int)
    cond3 = np.zeros((trial_count, TARGET_LIST_LENGTH), dtype=int)
    imputed_mask = np.zeros((trial_count, TARGET_LIST_LENGTH), dtype=int)
    lpp_mask = np.zeros((trial_count, TARGET_LIST_LENGTH), dtype=int)
    recalls = np.zeros((trial_count, TARGET_LIST_LENGTH), dtype=int)
    restricted_recalls = np.zeros((trial_count, TARGET_LIST_LENGTH), dtype=int)
    rec_itemids = np.zeros((trial_count, TARGET_LIST_LENGTH), dtype=int)
    for trial_index, ((participant, list_id), block) in enumerate(trial_groups):
        block = block.sort_values('PresentationOrder').reset_index(drop=True)
        subject[trial_index, 0] = int(participant)
        list_ids[trial_index, 0] = int(list_id)
        pres_itemids[trial_index, :] = block['item_id_code'].to_numpy(dtype=int)
        early_lpp[trial_index, :] = block['EarlyLPP'].to_numpy(dtype=float)
        late_lpp[trial_index, :] = block['LateLPP'].to_numpy(dtype=float)
        condition[trial_index, :] = block['condition'].to_numpy(dtype=int)
        cond3[trial_index, :] = block['cond3'].to_numpy(dtype=int)
        imputed_mask[trial_index, :] = block['imputed_mask'].to_numpy(dtype=int)
        lpp_mask[trial_index, :] = block['lpp_imputed'].to_numpy(dtype=int)
        recalled_positions = block.loc[block['Recalled'] == 1, 'PresentationOrder'].astype(int).tolist()
        recalled_item_ids = block.loc[block['Recalled'] == 1, 'item_id_code'].astype(int).tolist()
        for recall_index, position in enumerate(recalled_positions):
            recalls[trial_index, recall_index] = position
            restricted_recalls[trial_index, recall_index] = position
            rec_itemids[trial_index, recall_index] = recalled_item_ids[recall_index]
    return {
        'subject': subject,
        'list': list_ids,
        'listLength': list_length,
        'pres_itemnos': pres_itemnos,
        'pres_itemids': pres_itemids,
        'recalls': recalls,
        'restricted_recalls': restricted_recalls,
        'rec_itemids': rec_itemids,
        'EarlyLPP': early_lpp,
        'LateLPP': late_lpp,
        'condition': condition,
        'cond3': cond3,
        'lpp_imputed': lpp_mask,
    }


## Diagnostics


In [6]:
#| code-summary: Identify EEG rows flagged with presentation order ninety-nine
presentation_order_99 = (
    eeg_df[eeg_df['PresentationOrder'] == 99]
    .groupby(['Participant', 'List'])
    .size()
    .reset_index(name='count_presentationorder_99')
    .sort_values(['Participant', 'List'])
)
presentation_order_99


,Participant,List,count_presentationorder_99
0,202,7,1
1,203,7,1
2,204,7,1
3,205,7,1
4,206,7,1
5,207,7,1
6,210,7,1
7,211,7,1
8,212,7,1
9,213,7,1


In [7]:
#| code-summary: Prepare behavioral dataframe for diagnostics
behavior_df = prepare_behavioral_dataframe(BEHAVIOR_PATH)
merged_preview = merge_behavior_and_eeg(behavior_df, eeg_df)
behavior_df.head()


,Participant,List,TrialNumber,ThreeCondsCode,ConditionCode,ThreeConds,Recalled,PresentationOrder,Condition
0,202,1,3,1,1,Negative,1,1,Negative
1,202,1,4,2,2,Neutral,1,2,Neutral
2,202,1,5,1,1,Negative,1,3,Negative
3,202,1,6,1,1,Negative,0,4,Negative
4,202,1,7,3,2,Category,0,5,Neutral


In [8]:
#| code-summary: Summarize recall distribution
behavior_df['Recalled'].value_counts().to_dict()


{0: 3766, 1: 3074}

In [9]:
#| code-summary: Compare recalled events with and without EEG measurements
recalled_mask = behavior_df['Recalled'] == 1
lpp_missing_mask = merged_preview[['EarlyLPP', 'LateLPP']].isna().any(axis=1)

lpp_presence_summary = {
    'recalled_total': int(recalled_mask.sum()),
    'recalled_without_eeg': int((recalled_mask & lpp_missing_mask).sum()),
    'recalled_with_eeg': int((recalled_mask & ~lpp_missing_mask).sum()),
    'not_recalled_without_eeg': int((~recalled_mask & lpp_missing_mask).sum()),
    'not_recalled_with_eeg': int((~recalled_mask & ~lpp_missing_mask).sum()),
}
lpp_presence_summary


{'recalled_total': 3074,
 'recalled_without_eeg': 140,
 'recalled_with_eeg': 2934,
 'not_recalled_without_eeg': 231,
 'not_recalled_with_eeg': 3534}

In [10]:
#| code-summary: Recall rate by condition
behavior_df.groupby('Condition')['Recalled'].mean().to_dict()


{'Negative': 0.5322153574580759, 'Neutral': 0.40839527765631833}

## Dataset Preparation


In [11]:
#| code-summary: Factorize identifiers and impute LPP values
aligned_df = assign_img_placeholders(merged_preview)
aligned_df['item_id_code'], item_categories = factorize_series(aligned_df['ImgName'])
aligned_df['item_id_code'] = aligned_df['item_id_code'].astype(int)
aligned_df['condition'], condition_categories = factorize_series(
    aligned_df['Condition'], categories=['Negative', 'Neutral']
)
aligned_df['cond3'], cond3_categories = factorize_series(
    aligned_df['ThreeConds'], categories=['Negative', 'Neutral', 'Category']
)
aligned_df['condition'] = aligned_df['condition'].astype(int)
aligned_df['cond3'] = aligned_df['cond3'].astype(int)
imputed_df = impute_lpp_by_condition(aligned_df)
imputed_df.head()


,Participant,List,TrialNumber,ThreeCondsCode,ConditionCode,ThreeConds,Recalled,PresentationOrder,Condition,ImgName,EarlyLPP,LateLPP,imputed_mask,item_id_code,condition,cond3,lpp_imputed
0,202,1,3,1,1,Negative,1,1,Negative,neg139,-3.95610,-1.84030,0,465,1,1,0
1,202,1,4,2,2,Neutral,1,2,Neutral,neut2,0.66242,-0.23096,0,514,2,2,0
2,202,1,5,1,1,Negative,1,3,Negative,neg138,-0.56351,-0.30037,0,464,1,1,0
3,202,1,6,1,1,Negative,0,4,Negative,neg132,0.72071,0.91095,0,461,1,1,0
4,202,1,7,3,2,Category,0,5,Neutral,cat124,-0.64235,0.51120,0,18,2,3,0


In [12]:
#| code-summary: Assemble RecallDataset arrays and metadata summary
talmi_dataset = build_recall_dataset(imputed_df)
talmi_metadata = {
    'item_categories': item_categories,
    'condition_categories': condition_categories,
    'cond3_categories': cond3_categories,
}
trial_count = int(talmi_dataset['subject'].shape[0])
list_length = int(talmi_dataset['listLength'][0, 0])
recall_density = float(np.count_nonzero(talmi_dataset['recalls']) / talmi_dataset['recalls'].size)
restricted_recall_density = float(
    np.count_nonzero(talmi_dataset['restricted_recalls']) / talmi_dataset['restricted_recalls'].size
)
dataset_summary = {
    'trial_count': trial_count,
    'list_length': list_length,
    'recalls_shape': talmi_dataset['recalls'].shape,
    'recall_density': recall_density,
    'restricted_recall_density': restricted_recall_density,
    'lpp_imputed_items': int(talmi_dataset['lpp_imputed'].sum()),
}
save_dict_to_hdf5(talmi_dataset, f'{find_project_root()}/data/TalmiEEG.h5')
dataset_summary


{'trial_count': 342,
 'list_length': 20,
 'recalls_shape': (342, 20),
 'recall_density': 0.44941520467836255,
 'restricted_recall_density': 0.44941520467836255,
 'lpp_imputed_items': 371}

In [13]:
#| code-summary: (deprecated duplicate cleared)


In [14]:
talmi_dataset['lpp_imputed'].size

6840

In [15]:
subject_mask = generate_trial_mask(talmi_dataset, "data['subject'] == 202")
for key, value in talmi_dataset.items():
    print(f"{key}\n{value[subject_mask]}\n")

subject
[[202]
 [202]
 [202]
 [202]
 [202]
 [202]
 [202]
 [202]
 [202]]

list
[[1]
 [2]
 [3]
 [4]
 [5]
 [6]
 [7]
 [8]
 [9]]

listLength
[[20]
 [20]
 [20]
 [20]
 [20]
 [20]
 [20]
 [20]
 [20]]

pres_itemnos
[[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
 [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
 [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
 [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
 [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
 [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
 [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
 [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
 [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]]

pres_itemids
[[465 514 464 461  18  22  19 459 505   1  67  25 463  20 515 462 466  21
  513 460]
 [ 44 439 441 475 506   4  10 512 443 442 510 519  68 509   7   9 516   8
   12 438]
 [ 61  69  70 560 556 559  52 491  65 490 4

In [16]:
talmi_metadata

{'item_categories': ['cat1',
  'cat10',
  'cat102',
  'cat103',
  'cat104',
  'cat105',
  'cat106',
  'cat107',
  'cat110',
  'cat112',
  'cat114',
  'cat115',
  'cat116',
  'cat117',
  'cat12',
  'cat120',
  'cat122',
  'cat124',
  'cat126',
  'cat128',
  'cat129',
  'cat13',
  'cat130',
  'cat133',
  'cat135',
  'cat138',
  'cat141',
  'cat143',
  'cat145',
  'cat146',
  'cat147',
  'cat149',
  'cat15',
  'cat16',
  'cat17',
  'cat21',
  'cat22',
  'cat24',
  'cat27',
  'cat28',
  'cat33',
  'cat4',
  'cat41',
  'cat43',
  'cat44',
  'cat46',
  'cat47',
  'cat49',
  'cat51',
  'cat52',
  'cat54',
  'cat59',
  'cat62',
  'cat66',
  'cat70',
  'cat73',
  'cat8',
  'cat86',
  'cat88',
  'cat89',
  'cat9',
  'cat92',
  'cat93',
  'cat94',
  'cat95',
  'cat98',
  'missing_202_1_11',
  'missing_202_2_13',
  'missing_202_3_2',
  'missing_202_3_3',
  'missing_202_4_5',
  'missing_202_4_7',
  'missing_202_5_9',
  'missing_202_6_12',
  'missing_202_6_14',
  'missing_202_7_1',
  'missing_202_8_